## 1. Boilerplate

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.bracket.structure import load_bracket
from src.bracket.predictor import BracketPredictor, BracketPrediction
from src.simulation.mc_engine import MCSimulator, SimulationResult
from src.simulation.matchup_builder import TeamStatsLookup
from src.simulation.log5 import build_log5_win_prob_fn
from src.models.moe_ensemble import MOEEnsemble
from src.features.pipeline import build_features_for_split
from src.features.context_features import build_upset_rate_lookup
from src.features.kenpom_deltas import compute_deltas
from src.data.kaggle_loader import load_kenpom
from src.data.merge import merge_kenpom_with_matchups
from src.data.tournament_data import load_results, load_seeds, load_teams
from src.config import FIRST_YEAR, LAST_YEAR, SKIP_YEARS, MC_SIMULATIONS, TOURNAMENT_DIR

sns.set_theme(style="whitegrid")
logging.basicConfig(level=logging.INFO)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Load Bracket

In [ ]:
bracket = load_bracket(2025)

print(f"Slots:   {len(bracket.slots)}")
print(f"Teams:   {len(bracket.teams)}")
print(f"Rounds:  {sorted(bracket.round_slots.keys())}")
print()

# Show regions
regions = set()
for seed_str in bracket.teams:
    regions.add(seed_str[0])
print(f"Regions: {sorted(regions)}")
print()

# Display teams dict
teams_display = []
for seed_str, (tid, name, seed_num) in sorted(bracket.teams.items()):
    teams_display.append({"seed_str": seed_str, "team_id": tid, "name": name, "seed_num": seed_num})
teams_df = pd.DataFrame(teams_display)
teams_df

## 3. Train MOE + Build Lookup

In [ ]:
merged_df = merge_kenpom_with_matchups()
all_seasons = [y for y in range(FIRST_YEAR, LAST_YEAR + 1) if y not in SKIP_YEARS]
train_seasons = [y for y in all_seasons if y != 2025]

train_fs, test_fs = build_features_for_split(train_seasons, [2025], merged_df)
moe = MOEEnsemble()
moe.train_full_nested(train_fs, train_seasons, merged_df)

# Build lookup for simulation
kenpom_df = load_kenpom()
train_df = merged_df[merged_df["season"].isin(train_seasons)].copy()
train_deltas = compute_deltas(train_df)
upset_lookup = build_upset_rate_lookup(train_deltas)
lookup = TeamStatsLookup(2025, kenpom_df, upset_lookup)

print(f"Train seasons: {len(train_seasons)} ({train_seasons[0]}-{train_seasons[-1]})")
print(f"Teams in lookup: {len(lookup.team_stats)}")
print(f"Teams with seeds: {len(lookup.team_seeds)}")

## 4. Single Matchup Inspection

In [ ]:
# Pick 4 representative matchups from actual bracket teams
# Find teams by seed number within each region
seed_to_teams = {}  # seed_num -> list of (seed_str, team_id, name)
for seed_str, (tid, name, seed_num) in bracket.teams.items():
    seed_to_teams.setdefault(seed_num, []).append((seed_str, tid, name))

# Sort each list to get deterministic ordering
for sn in seed_to_teams:
    seed_to_teams[sn] = sorted(seed_to_teams[sn])

# Define matchups: (seed_a_num, idx_a, seed_b_num, idx_b, label)
matchup_defs = [
    (1, 0, 16, 0, "1 vs 16"),
    (5, 0, 12, 0, "5 vs 12"),
    (8, 0, 9, 0, "8 vs 9"),
    (1, 1, 1, 2, "Top seed vs Top seed"),
]

matchup_rows = []
for sa_num, ia, sb_num, ib, label in matchup_defs:
    teams_a = seed_to_teams.get(sa_num, [])
    teams_b = seed_to_teams.get(sb_num, [])
    if ia >= len(teams_a) or ib >= len(teams_b):
        continue
    _, tid_a, name_a = teams_a[ia]
    _, tid_b, name_b = teams_b[ib]
    if tid_a == tid_b:
        # For top seed vs top seed, pick different teams
        if len(teams_b) > ib + 1:
            _, tid_b, name_b = teams_b[ib + 1]
        else:
            continue

    round_num = 1 if sa_num != sb_num else 5  # F4 for same-seed matchups
    fs = lookup.get_matchup_features(tid_a, tid_b, round_num)
    # Fill NaN in gating features
    for col in fs.gating_features:
        if col in fs.X.columns:
            fs.X[col] = fs.X[col].fillna(0)
    decomp = moe.predict_decomposed(fs).iloc[0]

    matchup_rows.append({
        "matchup": label,
        "team_a": f"({sa_num}) {name_a}",
        "team_b": f"({sb_num}) {name_b}",
        "p_seed": decomp["p_seed"],
        "p_eff": decomp["p_eff"],
        "p_unc": decomp["p_unc"],
        "w_seed": decomp["w_seed"],
        "w_eff": decomp["w_eff"],
        "w_unc": decomp["w_unc"],
        "p_blend": decomp["p_blend"],
    })

matchup_df = pd.DataFrame(matchup_rows)
matchup_df.style.format({
    "p_seed": "{:.3f}", "p_eff": "{:.3f}", "p_unc": "{:.3f}",
    "w_seed": "{:.3f}", "w_eff": "{:.3f}", "w_unc": "{:.3f}",
    "p_blend": "{:.3f}",
})

## 5. Run MC Simulation

In [ ]:
def win_prob_fn(team_a, team_b, round_num):
    return lookup.get_win_prob(team_a, team_b, round_num, moe)

simulator = MCSimulator(n_simulations=10000, random_seed=42)
sim_result = simulator.simulate(bracket, win_prob_fn)

print(f"Simulations: {sim_result.n_simulations}")
print(f"Teams with championship prob > 0: {sum(1 for p in sim_result.championship_probs.values() if p > 0)}")
print(f"EV bracket picks: {len(sim_result.ev_bracket)}")

# Show top 5 championship contenders
top5 = sorted(sim_result.championship_probs.items(), key=lambda x: x[1], reverse=True)[:5]
for tid, prob in top5:
    name = lookup.team_names.get(tid, f"Team{tid}")
    seed_info = lookup.team_seeds.get(tid, ("?", "?"))
    print(f"  ({seed_info[1]}) {name}: {prob:.1%}")

## 6. Viz: Championship Probability Bar Chart

In [ ]:
# Top 16 teams by championship probability
top16 = sorted(sim_result.championship_probs.items(), key=lambda x: x[1], reverse=True)[:16]
top16_ids = [t[0] for t in top16]
top16_probs = [t[1] for t in top16]
top16_names = [lookup.team_names.get(tid, f"Team{tid}") for tid in top16_ids]
top16_seeds = [lookup.team_seeds.get(tid, ("?", 0))[1] for tid in top16_ids]

# Color by seed
seed_cmap = plt.cm.RdYlGn_r
norm = plt.Normalize(vmin=1, vmax=16)
colors = [seed_cmap(norm(s)) for s in top16_seeds]

fig, ax = plt.subplots(figsize=(10, 8))
labels = [f"({s}) {n}" for s, n in zip(top16_seeds, top16_names)]
y_pos = range(len(labels) - 1, -1, -1)
ax.barh(y_pos, top16_probs, color=colors, edgecolor="white", linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel("Championship Probability")
ax.set_title("MOE Championship Probabilities (Top 16 Teams)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

# Add value labels
for i, (y, prob) in enumerate(zip(y_pos, top16_probs)):
    ax.text(prob + 0.002, y, f"{prob:.1%}", va="center", fontsize=9)

sm = plt.cm.ScalarMappable(cmap=seed_cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, label="Seed", shrink=0.6)
cbar.set_ticks([1, 4, 8, 12, 16])

plt.tight_layout()
plt.show()

## 7. Viz: Advancement Probability Heatmap

In [ ]:
# Build advancement matrix: top 16 teams x rounds 1-6
round_labels = ["R64", "R32", "S16", "E8", "F4", "Champ"]
rounds = [1, 2, 3, 4, 5, 6]

adv_data = []
for tid in top16_ids:
    row = []
    team_adv = sim_result.advancement_probs.get(tid, {})
    for rnd in rounds:
        row.append(team_adv.get(rnd, 0.0))
    adv_data.append(row)

adv_matrix = pd.DataFrame(
    adv_data,
    index=[f"({lookup.team_seeds.get(t, ('?', 0))[1]}) {lookup.team_names.get(t, f'Team{t}')}" for t in top16_ids],
    columns=round_labels,
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    adv_matrix,
    annot=True,
    fmt=".0%",
    cmap="Greens",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Advancement Probabilities (Top 16 Teams)")
ax.set_ylabel("")
ax.set_xlabel("Round")

plt.tight_layout()
plt.show()

## 8. MOE vs Log5

In [ ]:
log5_fn = build_log5_win_prob_fn(2025)

sim_log5 = MCSimulator(n_simulations=10000, random_seed=42)
log5_result = sim_log5.simulate(bracket, log5_fn)

# Compare top teams
comparison_rows = []
for tid in top16_ids:
    name = lookup.team_names.get(tid, f"Team{tid}")
    seed = lookup.team_seeds.get(tid, ("?", 0))[1]
    moe_prob = sim_result.championship_probs.get(tid, 0.0)
    log5_prob = log5_result.championship_probs.get(tid, 0.0)
    comparison_rows.append({
        "team": f"({seed}) {name}",
        "moe_champ_prob": moe_prob,
        "log5_champ_prob": log5_prob,
        "diff": moe_prob - log5_prob,
    })

comp_df = pd.DataFrame(comparison_rows)
comp_df.style.format({
    "moe_champ_prob": "{:.1%}",
    "log5_champ_prob": "{:.1%}",
    "diff": "{:+.1%}",
}).bar(subset=["diff"], color=["#d65f5f", "#5fba7d"], align="zero")

## 9. Viz: MOE vs Log5 Championship Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

bar_height = 0.35
y_pos = np.arange(len(top16_ids))

# Reverse order so top team is at top
moe_probs = [sim_result.championship_probs.get(tid, 0.0) for tid in reversed(top16_ids)]
log5_probs = [log5_result.championship_probs.get(tid, 0.0) for tid in reversed(top16_ids)]
labels_rev = [
    f"({lookup.team_seeds.get(t, ('?', 0))[1]}) {lookup.team_names.get(t, f'Team{t}')}"
    for t in reversed(top16_ids)
]

ax.barh(y_pos + bar_height / 2, moe_probs, bar_height, label="MOE", color="#2196F3", edgecolor="white")
ax.barh(y_pos - bar_height / 2, log5_probs, bar_height, label="Log5", color="#FF9800", edgecolor="white")

ax.set_yticks(y_pos)
ax.set_yticklabels(labels_rev)
ax.set_xlabel("Championship Probability")
ax.set_title("MOE vs Log5: Championship Probabilities")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

## 10. Viz: MOE vs Log5 Advancement Curves

In [ ]:
# Top 2 championship contenders: advancement prob by round
top2_ids = top16_ids[:2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for idx, tid in enumerate(top2_ids):
    ax = axes[idx]
    name = lookup.team_names.get(tid, f"Team{tid}")
    seed = lookup.team_seeds.get(tid, ("?", 0))[1]

    moe_adv = sim_result.advancement_probs.get(tid, {})
    log5_adv = log5_result.advancement_probs.get(tid, {})

    moe_vals = [moe_adv.get(r, 0.0) for r in rounds]
    log5_vals = [log5_adv.get(r, 0.0) for r in rounds]

    ax.plot(round_labels, moe_vals, "o-", label="MOE", color="#2196F3", linewidth=2, markersize=7)
    ax.plot(round_labels, log5_vals, "s--", label="Log5", color="#FF9800", linewidth=2, markersize=7)

    ax.set_title(f"({seed}) {name}")
    ax.set_xlabel("Round")
    if idx == 0:
        ax.set_ylabel("Advancement Probability")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.set_ylim(0, 1.05)
    ax.legend()

plt.suptitle("MOE vs Log5: Advancement Curves (Top 2 Teams)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 11. Convergence Check

In [ ]:
sim_counts = [1000, 5000, 10000, 25000, 50000]
convergence = {n: {} for n in sim_counts}
top4_ids = sorted(
    sim_result.championship_probs,
    key=sim_result.championship_probs.get,
    reverse=True,
)[:4]

for n in sim_counts:
    print(f"Running {n} simulations...")
    sim_n = MCSimulator(n_simulations=n, random_seed=42)
    res_n = sim_n.simulate(bracket, win_prob_fn)
    for tid in top4_ids:
        convergence[n][tid] = res_n.championship_probs.get(tid, 0.0)

# Display convergence table
conv_rows = []
for n in sim_counts:
    row = {"n_sims": n}
    for tid in top4_ids:
        name = lookup.team_names.get(tid, f"Team{tid}")
        seed = lookup.team_seeds.get(tid, ("?", 0))[1]
        row[f"({seed}) {name}"] = convergence[n][tid]
    conv_rows.append(row)

conv_df = pd.DataFrame(conv_rows).set_index("n_sims")
conv_df.style.format("{:.2%}")

## 12. Viz: Convergence Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

palette = sns.color_palette("husl", len(top4_ids))

for i, tid in enumerate(top4_ids):
    name = lookup.team_names.get(tid, f"Team{tid}")
    seed = lookup.team_seeds.get(tid, ("?", 0))[1]
    probs = [convergence[n][tid] for n in sim_counts]
    ax.plot(sim_counts, probs, "o-", label=f"({seed}) {name}", color=palette[i], linewidth=2)

ax.set_xlabel("Number of Simulations")
ax.set_ylabel("Championship Probability")
ax.set_title("MC Convergence: Championship Probabilities")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(loc="best")

plt.tight_layout()
plt.show()

## 13. Sensitivity Analysis

In [ ]:
champ_id = max(sim_result.championship_probs, key=sim_result.championship_probs.get)
champ_name = lookup.team_names.get(champ_id, f"Team{champ_id}")
champ_seed = lookup.team_seeds.get(champ_id, ("?", 0))[1]
print(f"Sensitivity target: ({champ_seed}) {champ_name}")
print(f"Baseline championship prob: {sim_result.championship_probs[champ_id]:.1%}")

perturbations = [-0.10, -0.05, 0.0, 0.05, 0.10]
sensitivity = {}

for delta in perturbations:
    def perturbed_fn(a, b, r, d=delta):
        p = win_prob_fn(a, b, r)
        if a == champ_id or b == champ_id:
            if a == champ_id:
                p = np.clip(p + d, 0.01, 0.99)
            else:
                p = np.clip(p - d, 0.01, 0.99)
        return p

    sim_p = MCSimulator(10000, random_seed=42)
    res_p = sim_p.simulate(bracket, perturbed_fn)
    sensitivity[delta] = res_p.championship_probs.get(champ_id, 0.0)
    print(f"  delta={delta:+.0%}: champ_prob={sensitivity[delta]:.1%}")

## 14. Viz: Sensitivity Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

deltas = list(sensitivity.keys())
probs = list(sensitivity.values())
bar_labels = [f"{d:+.0%}" for d in deltas]

# Color bars: red for negative, green for positive, blue for baseline
bar_colors = []
for d in deltas:
    if d < 0:
        bar_colors.append("#e57373")
    elif d > 0:
        bar_colors.append("#81c784")
    else:
        bar_colors.append("#64b5f6")

bars = ax.bar(bar_labels, probs, color=bar_colors, edgecolor="white", linewidth=0.5)

# Add value labels
for bar, prob in zip(bars, probs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{prob:.1%}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.set_xlabel("Win Probability Perturbation")
ax.set_ylabel("Championship Probability")
ax.set_title(f"Sensitivity: ({champ_seed}) {champ_name} Championship Odds")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

plt.tight_layout()
plt.show()

## 15. Historical Validation

In [ ]:
# Load tournament results to find actual champions
tourney_results = load_results(TOURNAMENT_DIR)
seeds_df = load_seeds(TOURNAMENT_DIR)
teams_master = load_teams(TOURNAMENT_DIR)
id_to_name = dict(zip(teams_master["TeamID"], teams_master["TeamName"]))

def get_actual_champion(season):
    """Find the actual tournament champion for a season."""
    season_games = tourney_results[tourney_results["Season"] == season].copy()
    if season_games.empty:
        return None, None, None
    # Championship game is the last game (highest DayNum)
    champ_game = season_games.loc[season_games["DayNum"].idxmax()]
    champ_tid = int(champ_game["WTeamID"])
    champ_name = id_to_name.get(champ_tid, f"Team{champ_tid}")
    # Get seed
    seed_row = seeds_df[(seeds_df["Season"] == season) & (seeds_df["TeamID"] == champ_tid)]
    champ_seed = int(seed_row["SeedNum"].iloc[0]) if len(seed_row) > 0 else None
    return champ_tid, champ_name, champ_seed

# Validate on 3 seasons
val_seasons = [2018, 2023, 2024]
val_rows = []

for season in val_seasons:
    print(f"\n--- Season {season} ---")

    # Train MOE on all seasons except this one
    hist_train_seasons = [y for y in all_seasons if y != season]
    hist_train_fs, _ = build_features_for_split(hist_train_seasons, [season], merged_df)
    hist_moe = MOEEnsemble()
    hist_moe.train_full_nested(hist_train_fs, hist_train_seasons, merged_df)

    # Predict bracket
    predictor = BracketPredictor(hist_moe, n_simulations=10000, random_seed=42)
    prediction = predictor.predict_historical(season, merged_df=merged_df)

    # Get actual champion
    actual_tid, actual_name, actual_seed = get_actual_champion(season)

    # Get predicted champion
    pred_champ = prediction.champion
    pred_name = pred_champ.get("team_name", "?")
    pred_seed = pred_champ.get("seed", "?")
    pred_prob = pred_champ.get("probability", 0)

    # Find rank of actual champion in prediction
    sorted_champs = sorted(
        prediction.simulation.championship_probs.items(),
        key=lambda x: x[1],
        reverse=True,
    )
    actual_rank = None
    actual_pred_prob = 0.0
    for rank, (tid, prob) in enumerate(sorted_champs, 1):
        if tid == actual_tid:
            actual_rank = rank
            actual_pred_prob = prob
            break

    print(f"  Predicted: ({pred_seed}) {pred_name} [{pred_prob:.1%}]")
    print(f"  Actual:    ({actual_seed}) {actual_name} [predicted rank #{actual_rank}, {actual_pred_prob:.1%}]")

    val_rows.append({
        "season": season,
        "predicted_champion": f"({pred_seed}) {pred_name}",
        "predicted_prob": pred_prob,
        "actual_champion": f"({actual_seed}) {actual_name}",
        "actual_rank": actual_rank,
        "actual_pred_prob": actual_pred_prob,
        "correct": pred_name == actual_name,
    })

val_df = pd.DataFrame(val_rows)
val_df.style.format({
    "predicted_prob": "{:.1%}",
    "actual_pred_prob": "{:.1%}",
})